# Machine Learning Training Workflow

This notebook provides a complete workflow for training machine learning models using our advanced training pipeline. It includes data loading, preprocessing, model creation, training, evaluation, hyperparameter tuning, and model optimization.

## Setup Environment

First, we'll clone the repository and install the required dependencies.

In [ ]:
# Clone the repository
!git clone https://github.com/backup-bdg-6/datasets.git ml-training-pipeline
%cd ml-training-pipeline

In [ ]:
# Install required dependencies
!pip install -r requirements.txt

# Install additional dependencies for hyperparameter tuning and distributed training
!pip install "ray[tune]" deepspeed nlpaug

## Import Required Modules

Now, let's import all the necessary modules for our workflow.

In [ ]:
import os
import sys
import logging
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Union, Any, Tuple, Callable

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Import our modules
from src.data.loaders import DatasetLoader
from src.data.preprocessors import DataPreprocessor
from src.data.augmentation import TextAugmenter, augment_dataset
from src.model.architecture import create_model_from_config
from src.model.training import Trainer, TrainingArguments
from src.model.distributed_training import train_distributed, DeepSpeedConfig
from src.utils.tokenization import get_tokenizer
from src.utils.hyperparameter_tuning import optimize_hyperparameters
from src.utils.model_evaluation import evaluate_model
from src.model.optimization import optimize_model

## Configuration

Let's create a configuration for our training workflow. You can modify this configuration based on your specific needs.

In [ ]:
# Create a configuration directory if it doesn't exist
!mkdir -p config

In [ ]:
%%writefile config/colab_config.yaml
# Configuration for AI model training workflow in Colab/Kaggle

# Project information
project_name: "transformer_model_training"
output_dir: "./output"
seed: 42

# Model configuration
model:
  type: "decoder_only_transformer"
  size: "small"  # Using small model for Colab/Kaggle
  sizes:
    small:
      n_layers: 6
      n_heads: 8
      d_model: 512
      d_ff: 2048
      max_seq_length: 1024
    medium:
      n_layers: 12
      n_heads: 12
      d_model: 768
      d_ff: 3072
      max_seq_length: 2048
  dropout: 0.1
  attention:
    causal: true
    rotary_embedding: true

# Tokenizer configuration
tokenizer:
  type: "huggingface"
  name: "gpt2"
  vocab_size: 50257
  max_length: 1024
  padding_side: "right"
  truncation_side: "right"
  add_bos_token: true
  add_eos_token: true

# Training configuration
training:
  active_stage: "finetune"  # Using a smaller dataset for Colab/Kaggle
  stages:
    - name: "finetune"
      epochs: 3
      datasets:
        - name: "imdb"
          split: "train"
          streaming: false
          max_samples: 5000  # Limiting samples for Colab/Kaggle
        - name: "imdb"
          split: "test"
          streaming: false
          max_samples: 1000  # Limiting samples for Colab/Kaggle
      learning_rate:
        initial: 2.0e-5
        min: 1.0e-6
        schedule: "linear"
        warmup_steps: 200
  
  # Optimizer settings
  optimizer:
    name: "adamw"
    weight_decay: 0.01
    beta1: 0.9
    beta2: 0.999
    eps: 1.0e-8
    use_8bit: false
  
  # Mixed precision settings
  mixed_precision: "fp16"
  gradient_checkpointing: false
  gradient_clipping: 1.0
  
  # Checkpointing settings
  checkpointing:
    save_steps: 100
    keep_last_n: 2
    save_optimizer_state: true
  
  # Evaluation settings
  evaluation:
    eval_steps: 100
    early_stopping:
      enabled: true
      patience: 3
      metric: "eval_loss"
      mode: "min"

# Data processing configuration
data_processing:
  preprocessing:
    normalize_unicode: true
    normalize_whitespace: true
    remove_html: true
    min_length: 10
    max_length: 1024
  
  augmentation:
    enabled: true
    techniques:
      - name: "synonym_replacement"
        probability: 0.1
      - name: "random_deletion"
        probability: 0.05
  
  batching:
    train_batch_size: 8  # Smaller batch size for Colab/Kaggle
    eval_batch_size: 8
    gradient_accumulation_steps: 4
    dynamic_batching: false

# Hyperparameter optimization configuration
hyperparameter_optimization:
  enabled: false
  search_algorithm: "hyperopt"
  scheduler: "asha"
  num_samples: 5  # Reduced for Colab/Kaggle
  num_epochs: 1  # Reduced for Colab/Kaggle
  metric: "eval_loss"
  mode: "min"
  search_space:
    learning_rate:
      distribution: "log_uniform"
      min: 1.0e-5
      max: 1.0e-3
    weight_decay:
      distribution: "log_uniform"
      min: 1.0e-6
      max: 1.0e-2
    batch_size:
      distribution: "categorical"
      values: [4, 8, 16]
    dropout:
      distribution: "uniform"
      min: 0.1
      max: 0.5

# Model optimization configuration
model_optimization:
  quantization:
    enabled: false
    method: "dynamic"
    dtype: "int8"
  
  pruning:
    enabled: false
    method: "l1_unstructured"
    amount: 0.3

# Evaluation configuration
evaluation:
  metrics:
    - "loss"
    - "perplexity"
    - "accuracy"
    - "bleu"
    - "rouge"
  
  generation:
    max_length: 64
    num_beams: 4
    temperature: 1.0
    top_p: 0.9
    top_k: 50
    do_sample: true

# Logging configuration
logging:
  use_wandb: false
  wandb_project: "transformer_model_training"
  log_steps: 10
  save_logs: true

## Load Configuration

Now, let's load the configuration from the YAML file.

In [ ]:
def load_config(config_path: str) -> Dict:
    """Load configuration from YAML file."""
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    except Exception as e:
        logger.error(f"Error loading configuration: {e}")
        raise

# Load configuration
config_path = "config/colab_config.yaml"
config = load_config(config_path)

# Set random seed
torch.manual_seed(config['seed'])
np.random.seed(config['seed'])

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)

## Data Loading and Preprocessing

Let's load and preprocess the datasets for training and evaluation.

In [ ]:
# Get active stage
active_stage = config['training']['active_stage']
logger.info(f"Active training stage: {active_stage}")

# Get stage configuration
stage_config = None
for stage in config['training']['stages']:
    if stage['name'] == active_stage:
        stage_config = stage
        break

if stage_config is None:
    raise ValueError(f"Training stage {active_stage} not found in configuration")

# Initialize dataset loader
dataset_loader = DatasetLoader(config_path)

# Load datasets
train_datasets = []
for dataset_config in stage_config['datasets']:
    if dataset_config.get('split') == 'train':
        dataset = dataset_loader.load_dataset(
            dataset_config['name'],
            subset=dataset_config.get('subset'),
            split='train',
            streaming=dataset_config.get('streaming', False),
            max_samples=dataset_config.get('max_samples')
        )
        train_datasets.append(dataset)

eval_datasets = []
for dataset_config in stage_config['datasets']:
    if dataset_config.get('split') in ['validation', 'test']:
        dataset = dataset_loader.load_dataset(
            dataset_config['name'],
            subset=dataset_config.get('subset'),
            split=dataset_config.get('split'),
            streaming=dataset_config.get('streaming', False),
            max_samples=dataset_config.get('max_samples')
        )
        eval_datasets.append(dataset)

# Initialize data preprocessor
preprocessor = DataPreprocessor(config)

# Preprocess datasets
train_datasets = [preprocessor.process_dataset(ds) for ds in train_datasets]
eval_datasets = [preprocessor.process_dataset(ds) for ds in eval_datasets]

# Apply data augmentation if enabled
if config['data_processing']['augmentation']['enabled']:
    # Extract augmentation techniques and probabilities
    techniques = [t['name'] for t in config['data_processing']['augmentation']['techniques']]
    probabilities = [t['probability'] for t in config['data_processing']['augmentation']['techniques']]
    
    # Augment training datasets
    for i, dataset in enumerate(train_datasets):
        # Determine text column
        text_column = None
        if 'text' in dataset.column_names:
            text_column = 'text'
        elif 'content' in dataset.column_names:
            text_column = 'content'
        elif 'sentence' in dataset.column_names:
            text_column = 'sentence'
        
        if text_column:
            train_datasets[i] = augment_dataset(
                dataset=dataset,
                text_column=text_column,
                techniques=techniques,
                probabilities=probabilities,
                n_aug=1,
                keep_original=True
            )

# Display dataset information
print(f"Training dataset(s): {len(train_datasets)}")
for i, ds in enumerate(train_datasets):
    print(f"  Dataset {i+1}: {len(ds)} examples, columns: {ds.column_names}")

print(f"Evaluation dataset(s): {len(eval_datasets)}")
for i, ds in enumerate(eval_datasets):
    print(f"  Dataset {i+1}: {len(ds)} examples, columns: {ds.column_names}")

## Model Creation

Now, let's create the model and tokenizer based on the configuration.

In [ ]:
# Get tokenizer
tokenizer = get_tokenizer(config['tokenizer'])

# Create model
model = create_model_from_config(config)

# Print model information
print(f"Model type: {config['model']['type']}")
print(f"Model size: {config['model']['size']}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

## Hyperparameter Tuning (Optional)

If hyperparameter tuning is enabled in the configuration, we can run it to find the best hyperparameters for our model.

In [ ]:
if config['hyperparameter_optimization']['enabled']:
    logger.info("Running hyperparameter tuning")
    
    # Get hyperparameter optimization configuration
    hpo_config = config['hyperparameter_optimization']
    
    # Run hyperparameter optimization
    best_params = optimize_hyperparameters(
        config_path=config_path,
        search_space=None,  # Use default search space from config
        num_samples=hpo_config['num_samples'],
        num_epochs=hpo_config['num_epochs'],
        search_alg=hpo_config['search_algorithm'],
        scheduler=hpo_config['scheduler'],
        metric=hpo_config['metric'],
        mode=hpo_config['mode']
    )
    
    logger.info(f"Best hyperparameters: {best_params['best_config']}")
    
    # Load updated configuration
    with open(os.path.join(config['output_dir'], 'best_config', 'best_config.yaml'), 'r') as f:
        config = yaml.safe_load(f)
else:
    logger.info("Hyperparameter tuning is disabled")

## Model Training

Now, let's train the model using the Trainer class.

In [ ]:
# Create training arguments
training_args = TrainingArguments(config, active_stage)

# Create trainer
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_datasets[0] if train_datasets else None,
    eval_dataset=eval_datasets[0] if eval_datasets else None,
    args=training_args
)

# Train model
logger.info("Starting model training")
results = trainer.train()

logger.info(f"Training results: {results}")

## Model Evaluation

Let's evaluate the trained model using our comprehensive evaluation metrics.

In [ ]:
# Evaluate model
logger.info("Evaluating model")

# Get evaluation configuration
eval_config = config['evaluation']

# Evaluate model
metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    eval_dataset=eval_datasets[0] if eval_datasets else None,
    task_type="text_generation",
    metrics=eval_config['metrics'],
    batch_size=config['data_processing']['batching']['eval_batch_size'],
    max_length=eval_config['generation']['max_length'],
    num_beams=eval_config['generation']['num_beams'],
    temperature=eval_config['generation']['temperature'],
    top_p=eval_config['generation']['top_p'],
    top_k=eval_config['generation']['top_k'],
    do_sample=eval_config['generation']['do_sample'],
    output_dir=os.path.join(config['output_dir'], 'evaluation'),
    visualize=True,
    show_plots=True
)

logger.info(f"Evaluation metrics: {metrics}")

## Model Optimization (Optional)

If model optimization is enabled in the configuration, we can optimize the model for deployment.

In [ ]:
if config['model_optimization']['quantization']['enabled']:
    logger.info("Optimizing model with quantization")
    
    # Get optimization configuration
    opt_config = config['model_optimization']
    
    # Optimize model
    optimized_model = optimize_model(
        model=model,
        tokenizer=tokenizer,
        optimization_type=opt_config['quantization']['method'],
        output_dir=os.path.join(config['output_dir'], 'optimized_model'),
        save_model=True,
        benchmark=True
    )
    
    logger.info("Model optimization completed")
elif config['model_optimization']['pruning']['enabled']:
    logger.info("Optimizing model with pruning")
    
    # Get optimization configuration
    opt_config = config['model_optimization']
    
    # Optimize model
    optimized_model = optimize_model(
        model=model,
        tokenizer=tokenizer,
        optimization_type="pruning",
        pruning_config={
            "method": opt_config['pruning']['method'],
            "amount": opt_config['pruning']['amount'],
            "parameters_to_prune": ["weight"]
        },
        output_dir=os.path.join(config['output_dir'], 'optimized_model'),
        save_model=True,
        benchmark=True
    )
    
    logger.info("Model optimization completed")
else:
    logger.info("Model optimization is disabled")

## Text Generation with the Trained Model

Let's use our trained model to generate some text.

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=100, num_return_sequences=1, temperature=0.7, top_p=0.9, top_k=50):
    """Generate text using the trained model."""
    # Tokenize the prompt
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    
    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=num_return_sequences,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the generated text
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    
    return generated_texts

# Set the model to evaluation mode
model.eval()

# Generate text with different prompts
prompts = [
    "This movie was really",
    "The plot of the film",
    "I enjoyed watching"
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    generated_texts = generate_text(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_length=50,
        num_return_sequences=3,
        temperature=0.7,
        top_p=0.9,
        top_k=50
    )
    
    for i, text in enumerate(generated_texts):
        print(f"\nGenerated text {i+1}:\n{text}")

## Save and Load the Model

Let's demonstrate how to save and load the trained model.

In [ ]:
# Save the model and tokenizer
save_dir = os.path.join(config['output_dir'], 'final_model')
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model and tokenizer saved to {save_dir}")

In [ ]:
# Load the saved model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer

loaded_model = AutoModelForCausalLM.from_pretrained(save_dir)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)

print("Model and tokenizer loaded successfully")

# Generate text with the loaded model
prompt = "The best part of the movie was"
generated_texts = generate_text(
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    prompt=prompt,
    max_length=50,
    num_return_sequences=1
)

print(f"\nPrompt: {prompt}")
print(f"Generated text:\n{generated_texts[0]}")

## Distributed Training (Optional)

If you have access to multiple GPUs, you can use distributed training to speed up the training process. This is optional and will only work if you have multiple GPUs available.

In [ ]:
# Check if multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    
    # Create DeepSpeed configuration
    deepspeed_config = DeepSpeedConfig(
        zero_stage=2,
        offload_optimizer=False,
        offload_param=False,
        fp16=True,
        gradient_accumulation_steps=config['data_processing']['batching']['gradient_accumulation_steps'],
        gradient_clipping=config['training']['gradient_clipping'],
        output_dir=config['output_dir']
    )
    
    # Run distributed training
    # Note: This will only work if you run this notebook with the appropriate launcher
    # For example: python -m torch.distributed.launch --nproc_per_node=NUM_GPUS your_notebook.py
    results = train_distributed(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_datasets[0] if train_datasets else None,
        eval_dataset=eval_datasets[0] if eval_datasets else None,
        config_path=config_path,
        output_dir=config['output_dir'],
        num_epochs=stage_config['epochs'],
        batch_size=config['data_processing']['batching']['train_batch_size'],
        learning_rate=stage_config['learning_rate']['initial'],
        weight_decay=config['training']['optimizer']['weight_decay'],
        warmup_steps=stage_config['learning_rate']['warmup_steps'],
        gradient_accumulation_steps=config['data_processing']['batching']['gradient_accumulation_steps'],
        gradient_clipping=config['training']['gradient_clipping'],
        fp16=config['training']['mixed_precision'] == 'fp16',
        bf16=config['training']['mixed_precision'] == 'bf16',
        zero_stage=2,
        offload_optimizer=False,
        offload_param=False,
        use_deepspeed=True,
        local_rank=-1,  # This will be set by the launcher
        seed=config['seed']
    )
    
    print(f"Distributed training results: {results}")
else:
    print("Distributed training requires multiple GPUs")

## Conclusion

In this notebook, we've demonstrated the complete workflow for training machine learning models using our advanced training pipeline. The workflow includes:

1. Setting up the environment
2. Loading and preprocessing data
3. Creating and training a model
4. Evaluating the model
5. Optimizing the model (optional)
6. Generating text with the trained model
7. Saving and loading the model
8. Distributed training (optional)

You can customize this workflow by modifying the configuration file to suit your specific needs.